In [1]:
from pathlib import Path

from scipy.optimize import linear_sum_assignment
import torch

In [2]:
TORCH_SEED = 42
BATCH_SIZE = 32
NUM_CLASSES = 3  # background + rectangle + circle
MAX_NUM_OBJS = 10
IMAGE_SIZE = 128

BBOX_COST_WEIGHT = 1.0
CLASS_COST_WEIGHT = 2.0

In [3]:
x, target = torch.load("batch.pt", weights_only=False)
x.shape, target["boxes"].shape, target["labels"].shape, target["object_mask"].shape

(torch.Size([32, 3, 128, 128]),
 torch.Size([32, 2, 4]),
 torch.Size([32, 2]),
 torch.Size([32, 2]))

In [4]:
target_boxes = target["boxes"]
target_boxes_normalized = target_boxes / IMAGE_SIZE

target_labels = target["labels"]
object_mask = target["object_mask"]

In [5]:
torch.manual_seed(TORCH_SEED)

pred_locations = torch.rand(BATCH_SIZE, MAX_NUM_OBJS, 4)

pred_class_scores = torch.randn(BATCH_SIZE, MAX_NUM_OBJS, NUM_CLASSES)
pred_class_probs = torch.softmax(pred_class_scores, dim=-1)

pred_locations.shape, pred_class_probs.shape

(torch.Size([32, 10, 4]), torch.Size([32, 10, 3]))

# bbox cost

In [6]:
pred_locations.shape, pred_locations[0]

(torch.Size([32, 10, 4]),
 tensor([[0.8823, 0.9150, 0.3829, 0.9593],
         [0.3904, 0.6009, 0.2566, 0.7936],
         [0.9408, 0.1332, 0.9346, 0.5936],
         [0.8694, 0.5677, 0.7411, 0.4294],
         [0.8854, 0.5739, 0.2666, 0.6274],
         [0.2696, 0.4414, 0.2969, 0.8317],
         [0.1053, 0.2695, 0.3588, 0.1994],
         [0.5472, 0.0062, 0.9516, 0.0753],
         [0.8860, 0.5832, 0.3376, 0.8090],
         [0.5779, 0.9040, 0.5547, 0.3423]]))

In [7]:
target_boxes.shape, target_boxes[0]

(torch.Size([32, 2, 4]),
 tensor([[ 6., 50., 32., 74.],
         [ 4.,  4., 32., 34.]]))

In [8]:
# Compute pairwise costs against every padded target slot first.
bbox_cost = torch.cdist(pred_locations, target_boxes_normalized, p=2)
bbox_cost.shape

torch.Size([32, 10, 2])

# class cost

In [9]:
pred_class_probs.shape, pred_class_probs[0]

(torch.Size([32, 10, 3]),
 tensor([[0.1671, 0.1696, 0.6633],
         [0.3678, 0.5164, 0.1159],
         [0.6159, 0.2579, 0.1262],
         [0.2270, 0.4838, 0.2891],
         [0.1415, 0.8241, 0.0344],
         [0.2844, 0.3465, 0.3691],
         [0.3836, 0.5248, 0.0915],
         [0.0589, 0.1124, 0.8287],
         [0.5719, 0.2890, 0.1391],
         [0.6762, 0.1609, 0.1629]]))

In [10]:
target_labels.shape, target_labels[0]

(torch.Size([32, 2]), tensor([2, 2]))

In [11]:
import torch.nn.functional as F

target_labels_one_hot = F.one_hot(target_labels, num_classes=NUM_CLASSES).float()
target_labels_one_hot.shape, target_labels_one_hot[0]

(torch.Size([32, 2, 3]),
 tensor([[0., 0., 1.],
         [0., 0., 1.]]))

In [12]:
class_cost = torch.cdist(
    pred_class_probs,
    target_labels_one_hot,
    p=2,
)

class_cost.shape

torch.Size([32, 10, 2])

In [13]:
bbox_cost.min(), bbox_cost.max(), class_cost.min(), class_cost.max()

(tensor(0.1875), tensor(1.8310), tensor(0.0443), tensor(1.3343))

# agg cost

In [14]:
cost = bbox_cost * BBOX_COST_WEIGHT + class_cost * CLASS_COST_WEIGHT

In [15]:
cost.shape

torch.Size([32, 10, 2])

In [16]:
cost[0]

tensor([[1.8906, 2.2405],
        [2.6327, 3.0316],
        [3.3544, 3.3884],
        [2.7643, 2.9005],
        [3.4145, 3.6294],
        [1.8924, 2.2879],
        [2.6506, 2.5143],
        [1.4955, 1.3181],
        [3.0421, 3.3032],
        [3.0093, 3.2529]])

In [17]:
object_mask[0]

tensor([True, True])

In [ ]:
target_classes = torch.zeros((BATCH_SIZE, MAX_NUM_OBJS), dtype=torch.long)

for i in range(BATCH_SIZE):
    pred_indices, target_indices = linear_sum_assignment(cost[i].detach().cpu().numpy())
    pred_indices = torch.as_tensor(pred_indices, dtype=torch.long)
    target_indices = torch.as_tensor(target_indices, dtype=torch.long)

    valid_matches = object_mask[i, target_indices]
    matched_pred_indices = pred_indices[valid_matches]
    matched_target_indices = target_indices[valid_matches]

    target_classes[i, matched_pred_indices] = target_labels[i, matched_target_indices]

target_classes.shape

(torch.Size([32, 10]), tensor([2, 0, 0, 0, 0, 0, 0, 2, 0, 0]))

In [28]:
target_labels[1]

tensor([1, 0])

In [26]:
target_classes[1]

tensor([0, 0, 0, 1, 0, 0, 0, 0, 0, 0])

In [25]:
pred_class_probs[1]

tensor([[0.1132, 0.2271, 0.6597],
        [0.4137, 0.1027, 0.4836],
        [0.3452, 0.5128, 0.1420],
        [0.4328, 0.4666, 0.1005],
        [0.1772, 0.2542, 0.5686],
        [0.5064, 0.2530, 0.2406],
        [0.3483, 0.0743, 0.5774],
        [0.3642, 0.0782, 0.5576],
        [0.3802, 0.1809, 0.4389],
        [0.1948, 0.1836, 0.6216]])

In [24]:
object_mask[1]

tensor([ True, False])

In [29]:
class_cost[1]

tensor([[1.0224, 1.1283],
        [1.1001, 0.7670],
        [0.6138, 0.8437],
        [0.6942, 0.7413],
        [0.9545, 1.0320],
        [0.9340, 0.6046],
        [1.1452, 0.8739],
        [1.1372, 0.8492],
        [1.0041, 0.7808],
        [1.0445, 1.0336]])